# Theory of Mind (ToM) Benchmark in Google Colab

This notebook runs the Theory of Mind benchmark using T4 GPU resources on Google Colab.

## Setup Instructions:
1. Open this notebook in Google Colab: https://colab.research.google.com/
2. Upload this file to Colab
3. Go to Runtime → Change runtime type → Select T4 GPU
4. Run all cells sequentially

## Step 1: Mount Google Drive (Optional but Recommended)

Mount your Google Drive to save results persistently.

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create working directory in Drive
import os
WORK_DIR = '/content/drive/MyDrive/ToM_Benchmark'
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(f"{WORK_DIR}/results", exist_ok=True)
print(f"Working directory: {WORK_DIR}")

Mounted at /content/drive
Working directory: /content/drive/MyDrive/ToM_Benchmark


## Step 2: Verify GPU Availability

In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU device:", torch.cuda.get_device_name(0))
    print("GPU memory:", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")
else:
    print("WARNING: No GPU available! Please enable T4 GPU in Runtime settings.")

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU device: Tesla T4
GPU memory: 15.637086208 GB


## Step 3: Install Required Dependencies

In [3]:
# Install required packages
!pip install -q transformers torch accelerate bitsandbytes openai huggingface-hub

print("\nAll dependencies installed!")


All dependencies installed!


## Step 4: Upload Project Files

### Option A: Upload files manually
Use the file upload dialog in Colab to upload all Python files and data files.

### Option B: Clone from GitHub (if repo is on GitHub)

In [ ]:
# Option B: Clone from GitHub (replace with your actual repo URL)
# !git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git /content/ToM_Benchmark
# %cd /content/ToM_Benchmark

# For manual upload, use this cell to check files:
import os
print("Current directory:", os.getcwd())
print("Files in current directory:", os.listdir('.'))

## Step 5: Create Project Files (If Uploading Manually)

Run the cells below to create the necessary Python files. You can also skip this if you've uploaded all files.

In [4]:
# Create project directory structure
#import os

#os.makedirs('data', exist_ok=True)
#os.makedirs('res/subset', exist_ok=True)

#print("Directory structure created!")
#print("Please upload your data files (hitom_project_sub.json, etc.) to the 'data' folder")

Directory structure created!
Please upload your data files (hitom_project_sub.json, etc.) to the 'data' folder


In [22]:
import os

# Define the subfolders using your WORK_DIR
data_dir = os.path.join(WORK_DIR, 'data')
res_dir = os.path.join(WORK_DIR, 'res/subset')

# Create the directories
os.makedirs(data_dir, exist_ok=True)
os.makedirs(res_dir, exist_ok=True)

print("Directory structure created in Google Drive!")
print(f"Please upload your data files to: {data_dir}")

Directory structure created in Google Drive!
Please upload your data files to: /content/drive/MyDrive/ToM_Benchmark/data


In [4]:
import os

file_path = 'data/hitom_project_sub.json'

if os.path.exists(file_path):
    print(f"✅ Success! '{file_path}' has been uploaded.")
else:
    print(f"❌ File not found. Please upload 'hitom_project_sub.json' to the 'data' folder.")

✅ Success! 'data/hitom_project_sub.json' has been uploaded.


## Step 6: Define Core Model Functions for Colab (with Batch Support)

This cell contains the modified model.py functions that work with HuggingFace models on T4 GPU.
**NEW:** Includes batch processing functions for improved throughput (up to 3-5x speedup).

In [ ]:
"""
Model calling functions adapted for Google Colab with T4 GPU
Uses HuggingFace transformers for local model inference
Includes BATCH processing support for improved throughput
Also supports GGUF models from HuggingFace
"""

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Optional, List
import os
from huggingface_hub import hf_hub_download, list_repo_files

# Global cache for models to avoid reloading
_model_cache = {}
_tokenizer_cache = {}

def get_model_and_tokenizer(model_name: str, use_4bit: bool = True, gguf_filename: Optional[str] = None):
    """
    Load or retrieve cached model and tokenizer.
    Uses 4-bit quantization to fit larger models on T4 GPU.
    Supports both standard HuggingFace models and GGUF models.
    """
    if model_name in _model_cache:
        return _model_cache[model_name], _tokenizer_cache[model_name]

    print(f"Loading model: {model_name}...")

    # Check if this is a GGUF model
    is_gguf = "GGUF" in model_name.upper() or (gguf_filename and gguf_filename.endswith(".gguf"))

    if is_gguf:
        # For GGUF models, we need to find the actual GGUF file
        # First, try to list files in the repo and find a suitable GGUF file
        try:
            files = list_repo_files(model_name)
            gguf_files = [f for f in files if f.endswith(".gguf")]

            if not gguf_files:
                raise ValueError(f"No GGUF files found in repository {model_name}")

            # If a specific filename was requested, use it
            if gguf_filename and gguf_filename in gguf_files:
                selected_file = gguf_filename
            else:
                # Prefer Q4_K_M or Q4_K_S for good balance of quality and speed
                # Fall back to any Q4 variant, then Q5, then Q8, then any available
                priority_order = ["Q4_K_M", "Q4_K_S", "Q4_0", "Q4_1", "Q5_K_M", "Q5_K_S", "Q5_0", "Q5_1", "Q8_0"]
                selected_file = None
                for priority in priority_order:
                    candidates = [f for f in gguf_files if priority in f]
                    if candidates:
                        selected_file = candidates[0]
                        break
                if not selected_file:
                    selected_file = gguf_files[0]  # Fall back to first available

            print(f"  Using GGUF file: {selected_file}")

            # Download the GGUF file
            gguf_path = hf_hub_download(repo_id=model_name, filename=selected_file)
            print(f"  GGUF file path: {gguf_path}")

            # Load tokenizer from the base model or a compatible tokenizer
            # Try to infer base model name from the GGUF repo name
            base_model_name = model_name.replace("-GGUF", "").replace("-gguf", "")
            if "/" in base_model_name:
                # For repos like unsloth/Qwen3-1.7B-GGUF, try to get the original model
                tokenizer_name = base_model_name
            else:
                tokenizer_name = model_name

            try:
                tokenizer = AutoTokenizer.from_pretrained(
                    tokenizer_name,
                    trust_remote_code=True,
                    padding_side="left"
                )
            except Exception as e:
                print(f"  Could not load tokenizer from {tokenizer_name}, trying {model_name}...")
                tokenizer = AutoTokenizer.from_pretrained(
                    model_name,
                    trust_remote_code=True,
                    padding_side="left"
                )

            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            # Load model from GGUF file using from_single_file
            # Transformers 4.43+ supports loading GGUF files directly
            model = AutoModelForCausalLM.from_single_file(
                gguf_path,
                torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
                device_map="auto" if torch.cuda.is_available() else None,
                trust_remote_code=True,
            )

        except Exception as e:
            print(f"Error loading GGUF model: {e}")
            raise

    else:
        # Standard HuggingFace model loading
        # Configure quantization for T4 GPU (16GB VRAM)
        if use_4bit and torch.cuda.is_available():
            from transformers import BitsAndBytesConfig
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.float16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True,
            )
            dtype = torch.float16
        else:
            quantization_config = None
            dtype = torch.float16 if torch.cuda.is_available() else torch.float32

        tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True,
            padding_side="left"
        )

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map="auto" if torch.cuda.is_available() else None,
            trust_remote_code=True,
            quantization_config=quantization_config,
        )

    _model_cache[model_name] = model
    _tokenizer_cache[model_name] = tokenizer

    if torch.cuda.is_available():
        print(f"Model loaded! GPU memory used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    else:
        print("Model loaded! (CPU mode)")
    return model, tokenizer


def call_model_huggingface(prompt: str, model_name: str = "Qwen/Qwen3-1.7B", max_new_tokens: int = 1024) -> str:
    """
    Call a HuggingFace model using local inference on T4 GPU.
    model: e.g., "Qwen/Qwen3-1.7B", "Qwen/Qwen3-0.6B", "meta-llama/Llama-3.2-1B", etc.
    """
    model, tokenizer = get_model_and_tokenizer(model_name)

    messages = [
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return response.strip()


def call_model_huggingface_SoO(prompt: str, model_name: str = "Qwen/Qwen3-1.7B", max_new_tokens: int = 1024) -> str:
    """
    Call a HuggingFace model for SoO (Special case) using local pipeline.
    Uses a specialized system prompt for SoO method.
    """
    model, tokenizer = get_model_and_tokenizer(model_name)

    messages = [
        {"role": "system", "content": "You are an expert at understanding human communication. Please leverage the Information provided and choose the most probable answer to the question from the options. Output your final answer by strictly following this format: [A], [B], [C], or [D]"},
        {"role": "user", "content": prompt},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.0,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return response.strip()


# =============================================================================
# BATCH PROCESSING FUNCTIONS - For improved throughput
# =============================================================================


def call_model_huggingface_batch(
    prompts: List[str],
    model_name: str = "Qwen/Qwen3-1.7B",
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant",
    batch_size: int = 8
) -> List[str]:
    """
    Process multiple prompts in batches for improved GPU utilization.
    
    Args:
        prompts: List of user prompts to process
        model_name: HuggingFace model identifier
        max_new_tokens: Maximum new tokens to generate per sample
        system_message: System message for chat template
        batch_size: Number of samples to process per batch
    
    Returns:
        List of generated responses (one per prompt)
    """
    model, tokenizer = get_model_and_tokenizer(model_name)
    all_responses = []
    
    # Process in batches
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i:i + batch_size]
        
        # Build messages for each prompt in batch
        batch_messages = [
            [
                {"role": "system", "content": system_message},
                {"role": "user", "content": prompt},
            ]
            for prompt in batch_prompts
        ]
        
        # Apply chat template to all messages
        texts = [
            tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            for messages in batch_messages
        ]
        
        # Tokenize with padding for batch processing
        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            pad_to_multiple_of=8
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        
        # Decode each output, removing the input prompt portion
        for j, output in enumerate(outputs):
            input_length = inputs['input_ids'][j].shape[0]
            response = tokenizer.decode(output[input_length:], skip_special_tokens=True)
            all_responses.append(response.strip())
    
    return all_responses


def call_model_huggingface_batch_SoO(
    prompts: List[str],
    model_name: str = "Qwen/Qwen3-1.7B",
    max_new_tokens: int = 1024,
    batch_size: int = 8
) -> List[str]:
    """
    Batch version of SoO model caller with specialized system prompt.
    """
    system_message = "You are an expert at understanding human communication. Please leverage the Information provided and choose the most probable answer to the question from the options. Output your final answer by strictly following this format: [A], [B], [C], or [D]"
    return call_model_huggingface_batch(
        prompts=prompts,
        model_name=model_name,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        batch_size=batch_size
    )


def clear_model_cache():
    """Clear model cache to free GPU memory."""
    global _model_cache, _tokenizer_cache
    _model_cache.clear()
    _tokenizer_cache.clear()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("Model cache cleared!")

print("Model functions defined! (Now with BATCH processing support)")


Model functions defined!


## Step 7: Define Utility Functions

In [6]:
"""
Utility functions for data processing and evaluation
"""

import json
import re
import hashlib
from typing import Dict, List, Any, Optional

# =========================
# process choice text into dict
# =========================
def parse_choices(choice_text: str) -> Dict[str, str]:
    """
    Input:
        "A. blue_drawer, B. green_crate, C. red_bucket"
    Output:
        {"A": "blue_drawer", "B": "green_crate", "C": "red_bucket"}
    """
    pattern = r"([A-Z])\.\s*([^,]+)"
    matches = re.findall(pattern, choice_text)
    out = {}
    for k, v in matches:
        out[k.strip()] = v.strip()
    return out


def format_choices_for_prompt(choice_text: str) -> str:
    choices = parse_choices(choice_text)
    lines = [f"{k}. {v}" for k, v in choices.items()]
    return "\n".join(lines)

# =========================
# 1. load data
# =========================
def load_json(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if not isinstance(data, list):
        raise ValueError("The input file must be a JSON list.")

    return data


# =========================
# 2. normalize text and sample, and generate story id for deduplication
# =========================
def make_story_id(story: str) -> str:
    return hashlib.md5(story.strip().encode("utf-8")).hexdigest()


def normalize_text(s: str) -> str:
    return s.strip().lower().replace(" ", "_")


def normalize_sample(item: Dict[str, Any]) -> Dict[str, Any]:
    story = item["story"].strip()
    question = item["question"].strip()
    # Data files use "options" (a list) instead of "choices" (a string)
    options = item.get("options", [])
    choices = ", ".join(options) if isinstance(options, list) else item.get("choices", "").strip()
    answer = item["answer"].strip()

    return {
        "sample_id": item.get("sample_id"),
        "story_id": make_story_id(story),
        "prompting_type_raw": item.get("prompting_type"),
        "deception": item.get("deception"),
        "story_length": item.get("story_length"),
        "question_order": int(item.get("question_order")),
        "story": story,
        "question": question,
        "choices_raw": choices,
        "answer": answer,
    }

# =========================
# 3. parse model output/ prediction mapping
# =========================
def extract_option_letter(output_text: str) -> Optional[str]:
    """
    get an option letter A-O from model output, if any
    """
    text = output_text.strip()

    # case 1: just a single letter (VP prompt)
    if re.fullmatch(r"[A-O]", text, flags=re.IGNORECASE):
        return text.upper()

    # case 2: Answer: K (COTP prompt)
    m = re.search(r"answer\s*[:]\s*([A-O])\b", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # case 3: [A], [B], [C], [D] format (SoO)
    m = re.search(r"\[([A-O])\]", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    # case 4: the last appearing single letter
    matches = re.findall(r"\b([A-O])\b", text, flags=re.IGNORECASE)
    if matches:
        return matches[-1].upper()

    return None

def map_prediction_to_location(pred_raw: str, sample: Dict[str, Any]) -> Optional[str]:
    """
    map model output to standard choice name
    """
    choices = parse_choices(sample["choices_raw"])
    gold = normalize_text(sample["answer"])
    text = pred_raw.strip()

    # first try to extract option letter and map to choice value
    letter = extract_option_letter(text)
    if letter is not None and letter in choices:
        return normalize_text(choices[letter])

    # then try to directly output the location name
    text_norm = normalize_text(text)
    if text_norm == gold:
        return text_norm

    # if the model outputs a full sentence, check if any option value is present
    for _, value in choices.items():
        value_norm = normalize_text(value)
        if value_norm in text_norm:
            return value_norm

    return None

def judge_prediction(pred_raw: str, sample: Dict[str, Any]) -> Dict[str, Any]:
    gold = normalize_text(sample["answer"])
    pred_final = map_prediction_to_location(pred_raw, sample)

    return {
        "pred_raw": pred_raw,
        "pred_final": pred_final,
        "gold": gold,
        "correct": int(pred_final == gold) if pred_final is not None else 0,
    }

# =========================
# result analysis
# =========================
def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def report_accuracy_by_order(result_path: str) -> None:
    rows = load_jsonl(result_path)

    stats = {}
    for r in rows:
        order = r["question_order"]
        stats.setdefault(order, {"correct": 0, "total": 0})
        stats[order]["correct"] += int(r["correct"])
        stats[order]["total"] += 1

    print("\nAccuracy by question_order")
    for order in sorted(stats.keys()):
        c = stats[order]["correct"]
        t = stats[order]["total"]
        print(f"order {order}: {c}/{t} = {c/t:.4f}")

    # Overall accuracy
    total_correct = sum(s["correct"] for s in stats.values())
    total = sum(s["total"] for s in stats.values())
    print(f"\nOverall Accuracy: {total_correct}/{total} = {total_correct/total:.4f}")

print("Utility functions defined!")

Utility functions defined!


## Step 8: Define ToM Method Classes (with Batch Support)

In [7]:
"""
ToM Method implementations: SoO, PercepToM, DecomposeToM
Includes BATCH processing support for improved throughput
"""

import re
from typing import Dict, Any, Callable, Optional, List

# =========================
# SoO (Simulation of Other) Method
# =========================
class SoO:
    def __init__(self, llm_callable: Callable):
        self.llm_callable = llm_callable

    def extract_target_name(self, question: str) -> str:
        """
        Extracts the first character name whose perspective we need to adopt.
        Works for both simple and deeply nested recursive questions.
        Example: 'Where does Aiden think Carter thinks...' -> 'Aiden'
        """
        # Matches the first capitalized name immediately following 'Where does' or 'How does'
        match = re.search(r"(?:Where|How)\s+does\s+([A-Z][a-z]+)\b", question, re.IGNORECASE)
        if match:
            return match.group(1)

        return None

    def run(self, sample: Dict[str, Any]) -> str:
        # Fallback to 'options' if 'choices_raw' wasn't pre-normalized
        raw_choices = sample.get("choices_raw") or ", ".join(sample.get("options", []))
        choices_text = format_choices_for_prompt(raw_choices)

        name = self.extract_target_name(sample["question"])

        if name:
            prompt = f"""
            # Context
            {sample['story']}

            # Question
            {sample['question']}

            # Options
            {choices_text}

            Let's put ourselves in {name}'s shoes.
            """
        else:
            prompt = f"""
            # Context
            {sample['story']}

            # Question
            {sample['question']}

            # Options
            {choices_text}

            Let's think step-by-step.
            """

        # Pass the formatted prompt to the specialized LLM caller
        return self.llm_callable(prompt)

    @staticmethod
    def build_prompts_batch(samples: List[Dict[str, Any]]) -> List[str]:
        """
        Build prompts for a batch of samples (for batch processing).
        Returns a list of prompts that can be passed to the batch model caller.
        """
        prompts = []
        for sample in samples:
            raw_choices = sample.get("choices_raw") or ", ".join(sample.get("options", []))
            choices_text = format_choices_for_prompt(raw_choices)
            
            # Extract target name
            match = re.search(r"(?:Where|How)\s+does\s+([A-Z][a-z]+)\b", sample["question"], re.IGNORECASE)
            name = match.group(1) if match else None
            
            if name:
                prompt = f"""
            # Context
            {sample['story']}

            # Question
            {sample['question']}

            # Options
            {choices_text}

            Let's put ourselves in {name}'s shoes.
            """
            else:
                prompt = f"""
            # Context
            {sample['story']}

            # Question
            {sample['question']}

            # Options
            {choices_text}

            Let's think step-by-step.
            """
            prompts.append(prompt)
        return prompts


# =========================
# PercepToM Method
# =========================
class PercepToM:
    def __init__(self, llm_callable: Callable[[str], str]):
        self.llm_callable = llm_callable

    def extract_target_name(self, question: str) -> Optional[str]:
        """
        Extracts the character name whose mental state is being queried.
        Example: 'Where does Avery really think the lettuce is?' -> 'Avery'
        """
        # Pattern for "Where does {name} [think/believe/feel/know]..."
        match = re.search(r"Where does ([A-Z][a-z]+) (?:really )?(?:think|believe|feel|know|want)", question)
        if match:
            return match.group(1)

        # Pattern for "How does {name} think..."
        match = re.search(r"How does ([A-Z][a-z]+) think", question)
        if match:
            return match.group(1)

        return None

    def perception_inference(self, story: str, character: str) -> str:
        prompt = f"""Story:
        {story}

        Task: Based on the story above, identify exactly what the character '{character}' has perceived (seen, heard, or witnessed). If they left the room or were absent during certain events, explicitly note what they missed.

        Perception of {character}:"""
        return self.llm_callable(prompt)

    def perception_to_belief_inference(self, perception: str, character: str) -> str:
        prompt = f"""Character: {character}
        Perception: {perception}

        Task: Based *only* on the perception provided above (and strictly ignoring any omniscient knowledge of the actual world state), what does {character} currently believe to be true about the situation and the locations of objects/people?

        Belief State of {character}:"""
        return self.llm_callable(prompt)

    def answer_tom_question(self, story: str, belief: str, character: str, question: str, choices_text: str) -> str:
        prompt = f"""Story:
        {story}

        Belief State of {character}:
        {belief}

        Question:
        {question}

        Choices:
        {choices_text}

        Task: Answer the question. You must rely primarily on the "Belief State" of {character} to answer this question, rather than the objective reality described in the "Story".
        Think step by step, then give your final answer in the format:
        Answer: <option letter>"""
        return self.llm_callable(prompt)

    def run(self, sample: Dict[str, Any]) -> str:
        story = sample.get("story", sample.get("context", ""))
        question = sample["question"]
        choices_text = format_choices_for_prompt(sample["choices_raw"])

        # 1. Identify Target Character
        character = self.extract_target_name(question)

        # Fallback: If no character is identified (e.g. world-state question), do standard CoT
        if not character:
            fallback_prompt = f"""Story:
{story}

Question:
{question}

Choices:
{choices_text}

Think step by step, then give your final answer in the format:
Answer: <option letter>"""
            return self.llm_callable(fallback_prompt)

        # 2. PercepToM Pipeline
        perception = self.perception_inference(story, character)
        belief = self.perception_to_belief_inference(perception, character)
        final_answer = self.answer_tom_question(story, belief, character, question, choices_text)

        # Return final reasoning trace and answer
        return final_answer


# =========================
# DecomposeToM Method
# =========================
class DecomposeToM:
    def __init__(self, llm_callable, max_recursion_depth: int = 3):
        self.llm = llm_callable
        self.max_depth = max_recursion_depth

    def identify_subject(self, question: str) -> Optional[str]:
        prompt = f"""
        Analyze the following question and identify the specific agent or subject whose perspective,
        belief, or knowledge is being queried.
        If the question is about objective reality and not about someone's mental state, return "NONE".
        Respond ONLY with the name of the agent or "NONE".

        Question: "{question}"
        Agent:"""

        response = self.llm(prompt).strip()
        return None if "NONE" in response.upper() else response

    def reframe_question(self, question: str, subject: str) -> str:
        prompt = f"""
        Reframe the following Theory of Mind question to be a direct question from the perspective
        of the subject: {subject}. Remove references to their own belief.

        Original Question: "{question}"
        Reframed Direct Question:"""

        return self.llm(prompt).strip()

    def update_world_model(self, story: str, subject: str) -> str:
        prompt = f"""
        You are simulating the exact knowledge and memory of "{subject}".
        Read the following sequence of events. Based ONLY on what {subject} observed or was told,
        reconstruct the story. Omit any events, movements, or dialogue that {subject} is unaware of
        (e.g., things that happened before they entered a room or after they left).

        Original Story:
        {story}

        {subject}'s World Model (Observed Story):"""

        return self.llm(prompt).strip()

    def knowledge_availability(self, world_model: str, question: str) -> bool:
        prompt = f"""
        Based on the following known context, is it possible to answer the question?
        Return ONLY "YES" or "NO".

        Context:
        {world_model}

        Question: "{question}"
        Answer:"""

        response = self.llm(prompt).strip().upper()
        return "YES" in response

    def direct_qa(self, context: str, question: str) -> str:
        prompt = f"""
        Context: {context}
        Question: {question}
        Answer the question accurately based on the context provided.
        Answer:"""
        return self.llm(prompt).strip()

    def run(self, story: str, question: str, current_depth: int = 0) -> str:
        # Base case: Prevent infinite recursion
        if current_depth >= self.max_depth:
            return self.direct_qa(story, question)

        # 1. Subject Identification
        subject = self.identify_subject(question)

        # If no subjective perspective is queried, answer directly
        if not subject:
            return self.direct_qa(story, question)

        # 2. World Model Updation
        agent_world_model = self.update_world_model(story, subject)

        # 3. Question-Reframing
        reframed_question = self.reframe_question(question, subject)

        # 4. Knowledge Availability
        has_knowledge = self.knowledge_availability(agent_world_model, reframed_question)

        if not has_knowledge:
            return f"Based on the events, {subject} does not have enough information to know the answer."

        # Recursive Call for higher-order ToM
        return self.run(agent_world_model, reframed_question, current_depth + 1)

print("ToM Method classes defined!")

ToM Method classes defined!


## Step 9: Define Prompt Building Functions

In [8]:
"""
Prompt building functions for baseline methods
"""

def build_vp_prompt(sample: Dict[str, Any]) -> str:
    choices_text = format_choices_for_prompt(sample["choices_raw"])
    return f"""
            Story:
            {sample['story']}

            Question:
            {sample['question']}

            Choices:
            {choices_text}

            Please return exactly one uppercase option letter from A to O.
            Do not provide any explanation.
            Do not repeat the question.
            Do not output anything except the single letter."""


def build_cotp_prompt(sample: Dict[str, Any]) -> str:
    choices_text = format_choices_for_prompt(sample["choices_raw"])
    return f"""
            Story:
            {sample['story']}

            Question:
            {sample['question']}

            Choices:
            {choices_text}

            Think step by step, then give your final answer in the format:
            Answer: <option letter>"""


def build_prompt(sample: Dict[str, Any], method: str) -> str:
    """
    Build prompts for different ToM methods.
    """
    method = method.upper()
    if method == "VP":
        return build_vp_prompt(sample)
    elif method == "COTP":
        return build_cotp_prompt(sample)
    raise ValueError(f"invalid method: {method}")

print("Prompt building functions defined!")

Prompt building functions defined!


## Step 10: Define Main Benchmark Runner (with Batch Processing)

In [9]:
"""
Main benchmark runner functions
Includes BATCH processing support for significant speed improvements
"""

from pathlib import Path
from typing import Optional, List
import time

# Global configuration
CURRENT_MODEL = "Qwen/Qwen3-1.7B"  # Change this to use different models

# Model options suitable for T4 GPU:
# - "Qwen/Qwen3-0.6B" (smallest, fastest)
# - "Qwen/Qwen3-1.7B" (recommended balance)
# - "Qwen/Qwen3-4B" (with 4-bit quantization)
# - "meta-llama/Llama-3.2-1B"
# - "meta-llama/Llama-3.2-3B" (with 4-bit quantization)

def run_one_sample(sample: Dict[str, Any], method: str, model_name: str = None) -> Dict[str, Any]:
    """
    Run a single test sample with the specified method.
    """
    if model_name is None:
        model_name = CURRENT_MODEL

    method_upper = method.upper()

    # Use appropriate model caller based on method
    if method_upper == "SOO":
        llm_callable = lambda p: call_model_huggingface_SoO(p, model_name)
    else:
        llm_callable = lambda p: call_model_huggingface(p, model_name)

    # ---------------------------------------------------------
    # BRANCH: PercepToM
    # ---------------------------------------------------------
    if method_upper == "PERCEPTOM":
        try:
            tom_solver = PercepToM(llm_callable=llm_callable)
            output_text = tom_solver.run(sample)
            prompt = f"PercepToM Pipeline initiated for Question: {sample['question']}"
        except Exception as e:
            print(f"CRASH DETECTED in PercepToM: {e}")
            output_text = ""
            prompt = "Error during execution"

    # ---------------------------------------------------------
    # BRANCH: Decompose-ToM
    # ---------------------------------------------------------
    elif method_upper == "DTOM":
        try:
            tom_solver = DecomposeToM(llm_callable=llm_callable)
            raw_story = sample.get("story", sample.get("context", ""))
            raw_question = sample["question"]

            output_text = tom_solver.run(story=raw_story, question=raw_question)
            prompt = f"Decompose-ToM Pipeline initiated for:\nStory: {raw_story}\nQuestion: {raw_question}"
        except Exception as e:
            print(f"CRASH DETECTED in DToM: {e}")
            output_text = ""
            prompt = "Error during execution"

    # ---------------------------------------------------------
    # BRANCH: SoO
    # ---------------------------------------------------------
    elif method_upper == "SOO":
        try:
            tom_solver = SoO(llm_callable=llm_callable)
            output_text = tom_solver.run(sample)
            prompt = f"SoO Pipeline initiated for Question: {sample['question']}"
        except Exception as e:
            print(f"CRASH DETECTED in SoO: {e}")
            output_text = ""
            prompt = "Error during execution"

    # ---------------------------------------------------------
    # BRANCH: Baseline methods (VP, CoTP)
    # ---------------------------------------------------------
    else:
        prompt = build_prompt(sample, method=method)
        output_text = llm_callable(prompt)

    judged = judge_prediction(output_text, sample)

    return {
        "sample_id": sample["sample_id"],
        "story_id": sample["story_id"],
        "method": method,
        "question_order": sample["question_order"],
        "deception": sample.get("deception", None),
        "story_length": sample.get("story_length", None),
        "question": sample["question"],
        "answer": sample["answer"],
        "prompt": prompt,
        **judged,
    }


# =============================================================================
# BATCH PROCESSING FUNCTIONS
# =============================================================================

def run_batch_samples(
    samples: List[Dict[str, Any]],
    method: str,
    model_name: str,
    batch_size: int = 8
) -> List[Dict[str, Any]]:
    """
    Process multiple samples in a batch for improved throughput.
    Only works for single-call methods (VP, CoTP, SoO).
    Multi-call methods (PercepToM, DecomposeToM) fall back to single processing.
    
    Args:
        samples: List of normalized samples to process
        method: The ToM method to use
        model_name: Model identifier
        batch_size: Number of samples per batch
    
    Returns:
        List of result dictionaries
    """
    method_upper = method.upper()
    results = []
    
    # Methods that support true batching (single LLM call per sample)
    if method_upper in ["VP", "COTP", "SOO"]:
        # Build all prompts first
        prompts = []
        prompt_infos = []  # Track sample info for results
        
        for sample in samples:
            if method_upper == "SOO":
                prompt = SoO.build_prompts_batch([sample])[0]
            elif method_upper == "VP":
                prompt = build_vp_prompt(sample)
            else:  # COTP
                prompt = build_cotp_prompt(sample)
            
            prompts.append(prompt)
            prompt_infos.append({
                "sample_id": sample["sample_id"],
                "story_id": sample["story_id"],
                "question_order": sample["question_order"],
                "deception": sample.get("deception"),
                "story_length": sample.get("story_length"),
                "question": sample["question"],
                "answer": sample["answer"],
                "prompt": prompt
            })
        
        # Select batch caller based on method
        if method_upper == "SOO":
            batch_caller = lambda p: call_model_huggingface_batch_SoO(p, model_name, batch_size=batch_size)
        else:
            batch_caller = lambda p: call_model_huggingface_batch(p, model_name, batch_size=batch_size)
        
        # Process all prompts in batches
        outputs = batch_caller(prompts)
        
        # Judge predictions and build results
        for i, (output, info) in enumerate(zip(outputs, prompt_infos)):
            judged = judge_prediction(output, samples[i])
            results.append({
                **info,
                **judged
            })
    else:
        # Multi-call methods fall back to single processing
        for sample in samples:
            result = run_one_sample(sample, method=method, model_name=model_name)
            results.append(result)
    
    return results


def run_dataset(
    input_path: str,
    output_path: str,
    method: str = "VP",
    model_name: str = None,
    max_samples: Optional[int] = None,
    resume: bool = False,
    use_batching: bool = True,
    batch_size: int = 8,
) -> None:
    """
    Run the benchmark on a dataset.
    
    Args:
        input_path: Path to input JSON file
        output_path: Path to output JSONL file
        method: ToM method to use (VP, CoTP, SoO, PercepToM, DTOM)
        model_name: HuggingFace model identifier
        max_samples: Maximum number of samples to process (None for all)
        resume: Whether to resume from existing output file
        use_batching: Whether to use batch processing for supported methods
        batch_size: Number of samples per batch (for batching mode)
    """
    if model_name is None:
        model_name = CURRENT_MODEL

    raw_data = load_json(input_path)
    samples = [normalize_sample(x) for x in raw_data]

    if max_samples is not None:
        samples = samples[:max_samples]

    output_file = Path(output_path)
    output_file.parent.mkdir(parents=True, exist_ok=True)

    total = len(samples)
    start_index = 0
    correct = 0
    open_mode = "w"
    
    # Determine if we can use batching for this method
    can_batch = use_batching and method.upper() in ["VP", "COTP", "SOO"]

    if resume:
        if not output_file.exists():
            raise FileNotFoundError(f"Cannot resume: The benchmark file '{output_path}' does not exist.")

        existing_results = load_jsonl(str(output_file))
        if existing_results:
            last_sample_id = existing_results[-1]["sample_id"]

            # Recalculate previous correct answers
            correct = sum(int(r.get("correct", 0)) for r in existing_results)

            # Find the index of the last processed sample
            try:
                last_idx = next(i for i, s in enumerate(samples) if s["sample_id"] == last_sample_id)
                start_index = last_idx + 1
            except StopIteration:
                raise ValueError(f"Last sample_id '{last_sample_id}' from output not found in dataset.")

            if start_index >= total:
                print("The benchmark is already fully completed. Exiting.")
                return

            print(f"Resuming benchmark. Skipping {start_index} already processed samples.")
            print(f"Starting from sample_id '{samples[start_index]['sample_id']}'.")
            open_mode = "a"

    samples_to_run = samples[start_index:]

    print(f"\n{'='*60}")
    print(f"Starting benchmark with model: {model_name}")
    print(f"Method: {method}")
    print(f"Total samples: {total}, Starting from: {start_index}")
    print(f"Batch processing: {'ENABLED' if can_batch else 'DISABLED (method requires sequential calls)'}")
    if can_batch:
        print(f"Batch size: {batch_size}")
    print(f"Output file: {output_path}")
    print(f"{'='*60}\n")

    start_time = time.time()
    processed = 0

    with open(output_file, open_mode, encoding="utf-8") as f:
        if can_batch:
            # Process in batches for better GPU utilization
            for batch_start in range(0, len(samples_to_run), batch_size):
                batch_end = min(batch_start + batch_size, len(samples_to_run))
                batch = samples_to_run[batch_start:batch_end]
                
                try:
                    batch_results = run_batch_samples(batch, method, model_name, batch_size=len(batch))
                    
                    for result in batch_results:
                        correct += result["correct"]
                        f.write(json.dumps(result, ensure_ascii=False) + "\n")
                    
                    f.flush()
                    processed += len(batch)
                    
                    # Calculate and display progress
                    current_i = start_index + processed
                    elapsed = time.time() - start_time
                    samples_per_sec = processed / elapsed if elapsed > 0 else 0
                    eta_seconds = (total - current_i) / samples_per_sec if samples_per_sec > 0 else 0
                    
                    if current_i % 10 == 0 or batch_end >= len(samples_to_run):
                        print(
                            f"[{current_i}/{total}] "
                            f"batch_size={len(batch)} "
                            f"running_acc={correct / current_i:.4f} "
                            f"speed={samples_per_sec:.2f} samples/sec "
                            f"ETA={eta_seconds/60:.1f}min"
                        )
                        
                except Exception as e:
                    print(f"\nBatch error: {e}. Falling back to single processing for this batch...")
                    # Fall back to single processing
                    for sample in batch:
                        try:
                            result = run_one_sample(sample, method=method, model_name=model_name)
                        except Exception as e2:
                            print(f"Error on sample {sample['sample_id']}: {e2}")
                            result = {
                                "sample_id": sample["sample_id"],
                                "story_id": sample["story_id"],
                                "method": method,
                                "question_order": sample["question_order"],
                                "deception": sample.get("deception"),
                                "story_length": sample.get("story_length"),
                                "question": sample["question"],
                                "answer": sample["answer"],
                                "prompt": f"{method} batch fallback",
                                "pred_raw": None,
                                "pred_final": None,
                                "gold": normalize_text(sample["answer"]),
                                "correct": 0,
                                "error": str(e2),
                            }
                        
                        correct += result["correct"]
                        f.write(json.dumps(result, ensure_ascii=False) + "\n")
                        f.flush()
                        processed += 1
                        
                        current_i = start_index + processed
                        if current_i % 10 == 0:
                            print(f"[{current_i}/{total}] sample_id={sample['sample_id']} correct={result['correct']} running_acc={correct / current_i:.4f}")
        else:
            # Original single-sample processing for multi-call methods
            for i, sample in enumerate(samples_to_run, start=start_index + 1):
                try:
                    result = run_one_sample(sample, method=method, model_name=model_name)
                except Exception as e:
                    print(f"Error on sample {sample['sample_id']}: {e}")

                    # Fallback result on error
                    if method.upper() in ["VP", "COTP"]:
                        fallback_prompt = build_prompt(sample, method=method)
                    else:
                        fallback_prompt = f"{method} execution crashed before prompt generation."

                    result = {
                        "sample_id": sample["sample_id"],
                        "story_id": sample["story_id"],
                        "method": method,
                        "question_order": sample["question_order"],
                        "deception": sample.get("deception"),
                        "story_length": sample.get("story_length"),
                        "question": sample["question"],
                        "answer": sample["answer"],
                        "prompt": fallback_prompt,
                        "pred_raw": None,
                        "pred_final": None,
                        "gold": normalize_text(sample["answer"]),
                        "correct": 0,
                        "error": str(e),
                    }

                correct += result["correct"]
                f.write(json.dumps(result, ensure_ascii=False) + "\n")
                f.flush()  # Force write to disk

                if i % 10 == 0 or i == total:  # Print every 10 samples
                    elapsed = time.time() - start_time
                    samples_per_sec = i / elapsed if elapsed > 0 else 0
                    print(
                        f"[{i}/{total}] sample_id={sample['sample_id']} "
                        f"correct={result['correct']} "
                        f"running_acc={correct / i:.4f} "
                        f"speed={samples_per_sec:.2f} samples/sec"
                    )

    total_time = time.time() - start_time
    print(f"\n{'='*60}")
    print(f"Finished! Results saved to: {output_path}")
    print(f"Final Accuracy: {correct}/{total} = {correct / total:.4f}")
    print(f"Total time: {total_time/60:.2f} minutes")
    print(f"Average speed: {total/total_time:.2f} samples/sec")
    print(f"{'='*60}")

print("Main benchmark runner defined! (Now with BATCH processing support)")

Main benchmark runner defined!


## Step 11: Run the Benchmark

Choose your configuration and run the benchmark.

In [ ]:
# ============================================================
# CONFIGURATION - Modify these values for your run
# ============================================================

# Model selection (see options below)
MODEL_NAME = "Qwen/Qwen3-1.7B"  # Recommended for T4 GPU
# Other options:
# - "Qwen/Qwen3-0.6B" (fastest, smallest)
# - "Qwen/Qwen3-4B" (better quality, requires 4-bit quantization)
# - "meta-llama/Llama-3.2-1B"
# - "meta-llama/Llama-3.2-3B"

# Method selection
METHOD = "SoO"  # Options: "VP", "COTP", "SoO", "PercepToM", "DTOM"

# Data path
INPUT_PATH = "data/hitom_project_sub.json"

# Output path (save to Drive for persistence)
OUTPUT_PATH = f"/content/drive/MyDrive/ToM_Benchmark/results/hitom_cotp_results_{METHOD.lower()}_{MODEL_NAME.replace('/', '_')}.jsonl"

# Number of samples to run (set to None for all)
MAX_SAMPLES = 300  # Start with a small number to test

# Resume from previous run (if interrupted)
RESUME = False

# ============================================================
# BATCH PROCESSING CONFIGURATION (NEW!)
# ============================================================
# Batch processing significantly speeds up single-call methods
# (VP, CoTP, SoO) by processing multiple samples per forward pass.
# Multi-call methods (PercepToM, DecomposeToM) automatically
# fall back to sequential processing.
USE_BATCHING = True  # Enable batch processing for supported methods
BATCH_SIZE = 8       # Number of samples per batch (adjust based on GPU memory)
                     # - 4:  Safer for larger models or longer contexts
                     # - 8:  Good balance for Qwen 1.7B on T4 (default)
                     # - 16: Faster if you have enough VRAM

print("Configuration:")
print(f"  Model: {MODEL_NAME}")
print(f"  Method: {METHOD}")
print(f"  Input: {INPUT_PATH}")
print(f"  Output: {OUTPUT_PATH}")
print(f"  Max Samples: {MAX_SAMPLES}")
print(f"  Resume: {RESUME}")
print(f"  Batch Processing: {USE_BATCHING} (batch_size={BATCH_SIZE})")

Configuration:
  Model: Qwen/Qwen3-1.7B
  Method: SoO
  Input: data/hitom_project_sub.json
  Output: /content/drive/MyDrive/ToM_Benchmark/results/hitom_cotp_results_soo_Qwen_Qwen3-1.7B.jsonl
  Max Samples: 300
  Resume: False


In [ ]:
# ============================================================
# BATCH PROCESSING SPEED DEMONSTRATION (OPTIONAL)
# ============================================================
# This cell compares single vs batch processing speed.
# Run this before the full benchmark to verify batching works.

TEST_BATCH_SIZE = 4  # Small batch for quick test

def compare_processing_speed():
    """Compare single vs batch processing on a small subset."""
    print("="*60)
    print("BATCH PROCESSING SPEED COMPARISON")
    print("="*60)
    
    # Load test samples
    test_data = load_json(INPUT_PATH)
    test_samples = [normalize_sample(x) for x in test_data[:TEST_BATCH_SIZE]]
    
    print(f"\nTesting with {len(test_samples)} samples using method: {METHOD}")
    print(f"Model: {MODEL_NAME}\n")
    
    # Test single processing
    print("-"*40)
    print("SINGLE PROCESSING (one at a time):")
    print("-"*40)
    start_time = time.time()
    single_results = []
    for sample in test_samples:
        result = run_one_sample(sample, method=METHOD, model_name=MODEL_NAME)
        single_results.append(result)
    single_time = time.time() - start_time
    
    print(f"Time: {single_time:.2f} seconds")
    print(f"Speed: {len(test_samples)/single_time:.2f} samples/sec")
    
    # Test batch processing
    print("\n" + "-"*40)
    print("BATCH PROCESSING (all at once):")
    print("-"*40)
    start_time = time.time()
    batch_results = run_batch_samples(
        test_samples, 
        method=METHOD, 
        model_name=MODEL_NAME, 
        batch_size=TEST_BATCH_SIZE
    )
    batch_time = time.time() - start_time
    
    print(f"Time: {batch_time:.2f} seconds")
    print(f"Speed: {len(test_samples)/batch_time:.2f} samples/sec")
    
    # Compare
    print("\n" + "="*60)
    print("RESULTS:")
    print("="*60)
    speedup = single_time / batch_time
    print(f"Speedup: {speedup:.2f}x faster with batching")
    print(f"Time saved: {single_time - batch_time:.2f} seconds")
    
    # Verify results match
    matches = sum(1 for s, b in zip(single_results, batch_results) 
                  if s['pred_final'] == b['pred_final'])
    print(f"Result consistency: {matches}/{len(test_samples)} samples match")
    print("="*60)
    
    return speedup

# Uncomment to run comparison before full benchmark:
# compare_processing_speed()

In [12]:
# ============================================================
# LOAD DATA AND VERIFY
# ============================================================

# Check if data file exists
import os
if not os.path.exists(INPUT_PATH):
    print(f"ERROR: Data file not found: {INPUT_PATH}")
    print("\nAvailable files:")
    for root, dirs, files in os.walk('.'):
        for file in files:
            if file.endswith('.json'):
                print(f"  {os.path.join(root, file)}")
else:
    # Load and inspect data
    data = load_json(INPUT_PATH)
    print(f"Loaded {len(data)} samples from {INPUT_PATH}")

    # Show first sample
    if data:
        sample = data[0]
        print("\nFirst sample preview:")
        print(f"  Sample ID: {sample.get('sample_id')}")
        print(f"  Question: {sample.get('question', 'N/A')[:100]}...")
        print(f"  Answer: {sample.get('answer', 'N/A')}")
        print(f"  Options: {len(sample.get('options', []))} choices")

Loaded 300 samples from data/hitom_project_sub.json

First sample preview:
  Sample ID: 14
  Question: Where is the watermelon really?...
  Answer: blue_container
  Options: 15 choices


In [17]:
# ============================================================
# RUN THE BENCHMARK
# ============================================================

# Set the current model
CURRENT_MODEL = MODEL_NAME

# Run the benchmark with batch processing
run_dataset(
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
    method=METHOD,
    model_name=MODEL_NAME,
    max_samples=MAX_SAMPLES,
    resume=RESUME,
    use_batching=USE_BATCHING,
    batch_size=BATCH_SIZE,
)

Resuming benchmark. Skipping 121 already processed samples.
Starting from sample_id '627'.

Starting benchmark with model: Qwen/Qwen3-1.7B
Method: SoO
Total samples: 300, Starting from: 121
Output file: /content/drive/MyDrive/ToM_Benchmark/results/hitom_cotp_results_soo_Qwen_Qwen3-1.7B.jsonl

[130/300] sample_id=674 correct=0 running_acc=0.3308
[140/300] sample_id=834 correct=1 running_acc=0.3357
[150/300] sample_id=265 correct=1 running_acc=0.3333
[160/300] sample_id=294 correct=0 running_acc=0.3312


KeyboardInterrupt: 

## Step 12: Analyze Results

In [ ]:
# ============================================================
# ANALYZE RESULTS
# ============================================================

# Check if output file exists
if os.path.exists(OUTPUT_PATH):
    print("Analyzing results...\n")
    report_accuracy_by_order(OUTPUT_PATH)

    # Show some example predictions
    results = load_jsonl(OUTPUT_PATH)
    print("\n\nExample predictions (first 3):")
    for i, r in enumerate(results[:3]):
        print(f"\n--- Sample {r['sample_id']} ---")
        print(f"Question: {r['question'][:80]}...")
        print(f"Predicted: {r['pred_final']}")
        print(f"Gold: {r['gold']}")
        print(f"Correct: {bool(r['correct'])}")
else:
    print(f"No results file found at {OUTPUT_PATH}")

## Step 13: Download Results (Optional)

If you didn't mount Google Drive, download the results file.

In [ ]:
from google.colab import files

if os.path.exists(OUTPUT_PATH):
    files.download(OUTPUT_PATH)
    print("Download started!")
else:
    print("No results file to download")

## Additional: Test Single Sample

Use this section to test individual samples before running the full benchmark.

In [ ]:
# ============================================================
# TEST SINGLE SAMPLE
# ============================================================

# Load a single sample
test_data = load_json(INPUT_PATH)
test_sample = normalize_sample(test_data[0])

print("Testing with sample:")
print(f"Question: {test_sample['question']}")
print(f"Story: {test_sample['story'][:200]}...")
print(f"Options: {test_sample['choices_raw'][:100]}...")
print(f"Answer: {test_sample['answer']}")
print("\n" + "="*50 + "\n")

In [ ]:
# Test with VP method
result_vp = run_one_sample(test_sample, method="VP", model_name=MODEL_NAME)
print("VP Method Result:")
print(f"  Raw: {result_vp['pred_raw']}")
print(f"  Final: {result_vp['pred_final']}")
print(f"  Gold: {result_vp['gold']}")
print(f"  Correct: {bool(result_vp['correct'])}")

In [ ]:
# Test with SoO method
result_soo = run_one_sample(test_sample, method="SoO", model_name=MODEL_NAME)
print("\nSoO Method Result:")
print(f"  Raw: {result_soo['pred_raw']}")
print(f"  Final: {result_soo['pred_final']}")
print(f"  Gold: {result_soo['gold']}")
print(f"  Correct: {bool(result_soo['correct'])}")

In [ ]:
# Test with PercepToM method
result_perc = run_one_sample(test_sample, method="PercepToM", model_name=MODEL_NAME)
print("\nPercepToM Method Result:")
print(f"  Raw: {result_perc['pred_raw']}")
print(f"  Final: {result_perc['pred_final']}")
print(f"  Gold: {result_perc['gold']}")
print(f"  Correct: {bool(result_perc['correct'])}")

## System Information

In [ ]:
# Print system info for debugging
import platform
import psutil

print("System Information:")
print(f"Platform: {platform.platform()}")
print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"CPU Count: {psutil.cpu_count()}")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")